# Toy regression communication visualisation

This notebook runs the three-cluster DAC toy experiment for 30 clients and 50 rounds. It saves the complete replay timeline, final clients, and standard diagnostic plots under `Data_for_viz`.

In [1]:
import copy
import json
import os
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.classes import Client
from utils.toy_regression_utils import LinearRegression, generate_regression_multi
from utils.training_utils import client_information_exchange_DAC, train_clients_locally

SEED = 1
N_CLIENTS = 30
N_CLUSTERS = 3
CLIENTS_PER_CLUSTER = 10
N_ROUNDS = 50
N_DATA_TRAIN = 50
N_DATA_VAL = 100
BATCH_SIZE = 8
N_LOCAL_EPOCHS = 1
LEARNING_RATE = 0.003
N_NEIGHBORS = 5
TAU = 30
OUTPUT_DIR = "Data_for_viz"
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
print("Output directory:", os.path.abspath(OUTPUT_DIR))

ModuleNotFoundError: No module named 'torch'

In [ ]:
def make_dataset(theta, size, sigma=3):
    x, y = generate_regression_multi(theta, size, sigma)
    return torch.utils.data.TensorDataset(
        torch.from_numpy(x).float(),
        torch.from_numpy(y).float().reshape(-1, 1),
    )

thetas = [np.random.uniform(-10, 10, 10) for _ in range(N_CLUSTERS)]
trainsets = [
    make_dataset(thetas[cluster], N_DATA_TRAIN)
    for cluster in range(N_CLUSTERS)
    for _ in range(CLIENTS_PER_CLUSTER)
]
valsets = [make_dataset(theta, N_DATA_VAL) for theta in thetas]
initial_model = LinearRegression(10, 1)
clients = []
for client_id in range(N_CLIENTS):
    cluster = client_id // CLIENTS_PER_CLUSTER
    clients.append(Client(
        train_set=trainsets[client_id], val_set=valsets[cluster],
        idxs_train=None, idxs_val=None, criterion=torch.nn.MSELoss(),
        lr=LEARNING_RATE, device=torch.device("cpu"), batch_size=BATCH_SIZE,
        num_users=N_CLIENTS, model=copy.deepcopy(initial_model), idx=client_id,
        stopping_rounds=N_ROUNDS + 1, ratio=1 / N_CLUSTERS,
        dataset="toy_problem", shift=None, theta=thetas[cluster],
    ))
print(f"Created {len(clients)} clients in {N_CLUSTERS} clusters")

In [ ]:
def snapshot(clients, round_number, communications):
    return {
        "round": round_number,
        "clusterAverageValidationLoss": [
            float(np.mean([c.val_loss_list[-1] for c in clients if c.group == cluster]))
            for cluster in range(N_CLUSTERS)
        ],
        "nodes": [
            {"id": str(c.idx), "cluster": c.group, "validationLoss": float(c.val_loss_list[-1])}
            for c in clients
        ],
        "communications": communications,
    }

def communication_records(clients):
    records = []
    for client in clients:
        if not client.exchanges_every_round:
            continue
        for peer in client.exchanges_every_round[-1]:
            records.append({
                "source": str(client.idx), "target": str(int(peer)),
                "similarity": float(client.similarity_scores[peer]),
                "sameCluster": client.group == clients[peer].group,
            })
    return records

clients = train_clients_locally(clients, N_LOCAL_EPOCHS, verbose=False)
states = [snapshot(clients, 0, [])]
parameters = {
    "nbr_neighbors_sampled": N_NEIGHBORS, "prior_update_rule": "softmax",
    "similarity_metric": "inverse_training_loss", "tau": TAU,
    "cosine_alpha": 0.0, "mergatron": "chill",
    "aggregation_weighting": "trainset_size", "dataset": "toy_problem",
    "minmax": False,
}
for round_number in range(N_ROUNDS):
    clients = client_information_exchange_DAC(clients, parameters, verbose=False, round=round_number)
    communications = communication_records(clients)
    clients = train_clients_locally(clients, N_LOCAL_EPOCHS, verbose=False)
    states.append(snapshot(clients, round_number + 1, communications))

replay = {"experiment": "toy_regression_dac_30_clients", "parameters": parameters, "states": states}
with open(os.path.join(OUTPUT_DIR, "toy_regression_replay.json"), "w", encoding="utf-8") as file:
    json.dump(replay, file, indent=2)
with open(os.path.join(OUTPUT_DIR, "toy_regression_final_clients.pkl"), "wb") as file:
    pickle.dump(clients, file)
print(f"Saved {len(states)} states and {sum(len(s['communications']) for s in states)} communication events")

In [ ]:
rounds = np.array([state["round"] for state in states])
cluster_losses = np.array([state["clusterAverageValidationLoss"] for state in states])
plt.figure(figsize=(9, 5))
for cluster in range(N_CLUSTERS):
    plt.plot(rounds, cluster_losses[:, cluster], label=f"Cluster {cluster + 1}")
plt.xlabel("Round"); plt.ylabel("Average validation MSE"); plt.title("Average validation loss by cluster")
plt.legend(); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "average_validation_loss_by_cluster.png"), dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
for client in clients[:3]:
    plt.plot(client.val_loss_list, label=f"Client {client.idx} validation")
    plt.plot(client.train_loss_list, linestyle="--", alpha=0.7, label=f"Client {client.idx} training")
plt.xlabel("Local training step"); plt.ylabel("MSE"); plt.title("Training and validation loss for sample clients")
plt.legend(ncol=2); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_client_training_validation_loss.png"), dpi=160)
plt.show()

def heatmap(matrix, title, filename):
    plt.figure(figsize=(8, 7)); plt.imshow(matrix, cmap="hot", interpolation="nearest"); plt.colorbar()
    plt.title(title); plt.xlabel("Peer client"); plt.ylabel("Client"); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=160); plt.show()

similarity_matrix = np.array([client.similarity_scores for client in clients])
prior_matrix = np.array([client.priors for client in clients])
sampled_matrix = np.array([client.n_sampled for client in clients])
heatmap(similarity_matrix, "Final similarity scores", "final_similarity_heatmap.png")
heatmap(prior_matrix, "Final neighbor priors", "final_neighbor_prior_heatmap.png")
heatmap(sampled_matrix, "Communication count", "communication_count_heatmap.png")

The main validation signal is the cluster loss plot. Since this is regression, lower MSE means better performance; it is intentionally labelled as loss rather than accuracy. The heatmaps check that the final similarity/prior structure and observed communication are sensible before exporting the data to the website.